In [ ]:
import json
import pandas as pd

def standardize_risk(risk_val):
    if not isinstance(risk_val, str):
        return risk_val
    risk_upper = risk_val.upper()
    if 'HIGH' in risk_upper:
        return 'High'
    elif 'MED' in risk_upper:
        return 'Medium'
    elif 'LOW' in risk_upper:
        return 'Low'
    elif 'NOT APPLICABLE' in risk_upper:
        return 'Not Applicable'
    return risk_val

In [ ]:
with open('../results_batch_1.json', 'r') as f:
    batch_1 = json.load(f)

data_1 = []
for paper_no, domains in batch_1.items():
    for domain, content in domains.items():
        # Handle the nested 'result' key for domains and the direct string for 'overall'
        result = content['result'] if isinstance(content, dict) else content
        data_1.append({'Paper no.': paper_no, 'Bias domain': domain, 'result': result})

df_1 = pd.DataFrame(data_1)
df_1['result'] = df_1['result'].apply(standardize_risk)
df_1 = df_1.sort_values(by=['Paper no.', 'Bias domain'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)
display(df_1[['Paper no.', 'Bias domain', 'result']].head())

In [ ]:
with open('../results_batch_2.json', 'r') as f:
    batch_2 = json.load(f)

data_2 = []
for paper_no, domains in batch_2.items():
    for domain, content in domains.items():
        result = content['result'] if isinstance(content, dict) else content
        data_2.append({'Paper no.': paper_no, 'Bias domain': domain, 'result': result})

df_2 = pd.DataFrame(data_2)
df_2['result'] = df_2['result'].apply(standardize_risk)
df_2 = df_2.sort_values(by=['Paper no.', 'Bias domain'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)
display(df_2[['Paper no.', 'Bias domain', 'result']].head())

In [ ]:
df_expert_wide = pd.read_csv('../data/sample_papers/sample_papers_expert_ratings_wide.csv')

df_expert = pd.melt(
    df_expert_wide, 
    id_vars=['Paper no.'], 
    var_name='Bias domain', 
    value_name='result'
)

# Clean up the 'Bias domain' column to match previous tables (just the number) and standardize types
df_expert['Bias domain'] = df_expert['Bias domain'].str.replace('Bias Domain ', '')
df_expert['result'] = df_expert['result'].apply(standardize_risk)
df_expert['Paper no.'] = df_expert['Paper no.'].astype(str)

df_expert = df_expert.sort_values(by=['Paper no.', 'Bias domain'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)
display(df_expert[['Paper no.', 'Bias domain', 'result']].head(10))

In [ ]:
with open('../domain_shot_results_batch_1.json', 'r') as f:
    domain_shot_data = json.load(f)

data_domain = []
for paper_key, domains in domain_shot_data.items():
    paper_no = paper_key.replace('.md', '')
    for domain_key, content in domains.items():
        if domain_key == 'overall_risk':
            domain = 'overall'
            result = content
            data_domain.append({'Paper no.': paper_no, 'Bias domain': domain, 'result': result})
        elif domain_key.startswith('domain_'):
            domain = domain_key.replace('domain_', '')
            result = pd.NA
            if isinstance(content, dict):
                parsed = content.get('parsed_response', {})
                if 'final_risk' in content:
                    result = content['final_risk']
                elif 'final_risk' in parsed:
                    result = parsed['final_risk']
                elif 'decision_path' in parsed and isinstance(parsed.get('decision_path'), list) and len(parsed['decision_path']) > 0:
                    result = parsed['decision_path'][-1]
            data_domain.append({'Paper no.': paper_no, 'Bias domain': domain, 'result': result})

df_domain = pd.DataFrame(data_domain)
df_domain['result'] = df_domain['result'].apply(standardize_risk)
df_domain = df_domain.sort_values(by=['Paper no.', 'Bias domain'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)

display(df_domain[['Paper no.', 'Bias domain', 'result']].head(15))

In [ ]:
df_1_r = df_1[df_1['Bias domain'] != 'overall'].rename(columns={'result': 'Result_Batch_1'})
df_2_r = df_2[df_2['Bias domain'] != 'overall'].rename(columns={'result': 'Result_Batch_2'})
df_expert_r = df_expert[df_expert['Bias domain'] != 'overall'].rename(columns={'result': 'Result_Expert'})
df_domain_r = df_domain[df_domain['Bias domain'] != 'overall'].rename(columns={'result': 'Result_Domain_Shot'})

df_wide = pd.merge(df_expert_r, df_1_r, on=['Paper no.', 'Bias domain'], how='outer')
df_wide = pd.merge(df_wide, df_2_r, on=['Paper no.', 'Bias domain'], how='outer')
df_wide = pd.merge(df_wide, df_domain_r, on=['Paper no.', 'Bias domain'], how='outer')

df_wide = df_wide.sort_values(by=['Paper no.', 'Bias domain'], key=lambda col: pd.to_numeric(col, errors='coerce')).reset_index(drop=True)
with pd.option_context('display.max_rows', None):
    display(df_wide)

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Merge LLM and expert dataframes to align them exactly for comparison
merged_1 = pd.merge(df_1, df_expert, on=['Paper no.', 'Bias domain'], suffixes=('_llm', '_expert'))
merged_2 = pd.merge(df_2, df_expert, on=['Paper no.', 'Bias domain'], suffixes=('_llm', '_expert'))
merged_1_2 = pd.merge(df_1, df_2, on=['Paper no.', 'Bias domain'], suffixes=('_1', '_2'))
merged_domain = pd.merge(df_domain, df_expert, on=['Paper no.', 'Bias domain'], suffixes=('_domain', '_expert'))

def print_agreement(df_merged, col1, col2, comparison_name):
    df_clean = df_merged.dropna(subset=[col1, col2])
    pct_agree = (df_clean[col1] == df_clean[col2]).mean() * 100
    kappa = cohen_kappa_score(df_clean[col1].astype(str), df_clean[col2].astype(str))
    print(f"{comparison_name}:")
    print(f"  Percentage Agreement: {pct_agree:.2f}%")
    print(f"  Cohen's Kappa:        {kappa:.4f}\n")

print_agreement(merged_1, 'result_llm', 'result_expert', "Batch 1 vs Expert")
print_agreement(merged_2, 'result_llm', 'result_expert', "Batch 2 vs Expert")
print_agreement(merged_1_2, 'result_1', 'result_2', "Batch 1 vs Batch 2")
print_agreement(merged_domain, 'result_domain', 'result_expert', "Domain Shot vs Expert")